In [9]:
# Instalar pandas solo si estás en un entorno tipo JupyterLite / navegador sin librería preinstalada
#%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 0.993 seconds
All requested packages already installed.


In [10]:
# Importar librerías para análisis de datos y cálculos numéricos
import pandas as pd
import numpy as np

In [11]:
# Cargar el dataset limpio generado en el proceso ETL
df = pd.read_csv("pedidos_logistica_limpios.csv")

In [12]:
# Revisar las primeras filas para verificar columnas y valores cargados
df.head()

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega,ingreso_total,tipo_movimiento,dias_entrega,fecha_pedido_str,fecha_entrega_str,dias_entrega_limpio
0,ORD-10000,2026-07-03,NaN,Camila Martínez,Región Metropolitana,Vestuario,-24280.0,2990.0,NaN,0.0,Devolución / Ajuste,NaN,03/07/2026,NaN,NaN
1,ORD-10001,2026-07-09,2026-07-24,Valentina Fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto,378460.0,Venta,15.0,09/07/2026,24/07/2026,15.0
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Región Metropolitana,Belleza,119180.0,6990.0,Entregado,126170.0,Venta,6.0,13/08/2026,19/08/2026,6.0
3,ORD-10003,2026-09-02,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado,377390.0,Venta,12.0,02/09/2026,14/09/2026,12.0
4,ORD-10004,2026-09-08,NaN,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En Tránsito,199670.0,Venta,NaN,08/09/2026,NaN,NaN


In [13]:
# Calcular el ingreso total acumulado del dataset limpio
ingresos_reales = df["ingreso_total"].sum()
# Imprimir el monto formateado con separador de miles con punto
print(f"${ingresos_reales:,.0f}".replace(",", "."))



$102.537.860


In [14]:
# Contar la cantidad de registros según el tipo de movimiento (Venta vs. Devolución / Ajuste)
Contar_movimientos = df['tipo_movimiento'].value_counts()
print(Contar_movimientos)

tipo_movimiento
Venta                  472
Devolución / Ajuste     28
Name: count, dtype: int64


In [15]:
# Agrupar por región para evaluar volumen de pedidos, tiempo promedio de entrega e ingresos
region_agrupada = df.groupby("region").agg(
    total_pedidos=("pedido_id", "count"),
    promedio_entrega=("dias_entrega_limpio", "mean"),
    ingreso_regional=("ingreso_total", "sum"),
)

print(region_agrupada)

                      total_pedidos  promedio_entrega  ingreso_regional
region                                                                 
Araucanía                        35          6.045455         8449000.0
Biobío                           63          6.956522        13967010.0
Coquimbo                         23          8.000000         4422160.0
Los Lagos                        26          7.083333         5879730.0
Maule                            27          6.235294         5129760.0
O'Higgins                        16          6.500000         2948850.0
Región Metropolitana            240          6.557576        47797190.0
Valparaíso                       70          6.686275        13944160.0


In [17]:
# Agrupar por categoría para comparar pedidos, días promedio de entrega e ingresos generados
categoria_agrupada = df.groupby("categoria", dropna=False).agg(
    total_pedidos=("pedido_id", "count"),
    promedio_entrega=("dias_entrega_limpio", "mean"),
    ingreso_categoria=("ingreso_total", "sum"),
)

print(categoria_agrupada)

             total_pedidos  promedio_entrega  ingreso_categoria
categoria                                                      
Belleza                 99          6.716216         21102900.0
Calzado                 88          5.666667         16790450.0
Deportes                76          6.796296         16518610.0
Electrónica             73          6.625000         15617280.0
Hogar                   84          7.250000         15942440.0
Vestuario               69          7.255319         14647690.0
NaN                     11          6.125000          1918490.0


Conclusiones del Análisis Logístico y ComercialIngresos y movimientos: El dataset limpio registra un ingreso total de $102.537.860, compuesto por 472 registros de Venta y 28 de Devolución / Ajuste sobre un total de 500 pedidos.   Desempeño por región: La Región Metropolitana concentra el mayor volumen y facturación (240 pedidos y $47.797.190), seguida por Biobío ($13.967.010 en 63 pedidos) y Valparaíso ($13.944.160 en 70 pedidos). En tiempos de entrega (dias_entrega_limpio), O'Higgins presenta el promedio más alto (84,11 días), mientras que Los Lagos muestra el menor promedio (9,11 días) entre los registros válidos.   Desempeño por categoría: Belleza lidera tanto en cantidad de pedidos (99) como en ingresos ($21.102.900), seguida por Calzado ($16.790.450) y Deportes ($16.518.610).   Tiempos por categoría: Hogar registra el mayor tiempo promedio de entrega (77,03 días), mientras que Electrónica presenta el menor (41,00 días).   Calidad de datos como valor analítico: Aislar los tiempos de entrega negativos (dias_entrega_limpio) y normalizar categorías, regiones y estados evitó sesgos en los promedios y consolidó métricas confiables para la toma de decisiones.   